# 05 — What-if Simulator: Counterfactual Predictions for Operators

Take the trained model from notebook 02 and answer operator questions like:

* *"¿Qué pasa con el % Iron Concentrate si cambio el pH de 9.4 a 9.8?"*
* *"¿Y si subo el flujo de amina 10%?"*

Two evaluators:

* **Naive** — substitutes the new sensor value and nudges rolling means proportionally. O(1), fast, approximate. Good for real-time UI sliders.
* **Exact** — replays the recent window of raw sensor data with the override applied and re-runs the rolling pipeline. Slower, accurate. Good for committed decisions.

Both return baseline + counterfactual + Δ for each target.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import mlflow
import mlflow.lightgbm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from frothiq.data.loader import SENSOR_COLS, TARGET_COLS, load_flotation
from frothiq.features.pipeline import FeatureConfig, build_features, list_feature_cols
from frothiq.models.whatif.simulator import (
    apply_overrides_naive,
    apply_overrides_exact,
    simulate_whatif_naive,
    simulate_whatif_exact,
)

mlflow.set_tracking_uri(f'file:{(ROOT / "mlruns").as_posix()}')

C:\Users\jona2\frothiq\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the trained LightGBM model

Pull the most recent run from MLflow.

In [2]:
target = 'pct_iron_concentrate'
model = None
for experiment_name in ['frothiq-baseline-fresh', 'frothiq-baseline']:
    runs = mlflow.search_runs(
        experiment_names=[experiment_name],
        filter_string=f"params.target = '{target}'",
        order_by=['start_time DESC'],
        max_results=1,
    )
    if len(runs):
        model = mlflow.lightgbm.load_model(f"runs:/{runs.iloc[0]['run_id']}/model")
        print(f'Loaded model for {target} from {experiment_name}')
        break
if model is None:
    raise RuntimeError(f'No run found for {target}. Run notebook 02 (or 02b) first.')

C:\Users\jona2\frothiq\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Loaded model for pct_iron_concentrate from frothiq-baseline-fresh


## 2. Pick a recent observation as the operator's current state

We use the last row of the gold table (most recent feature vector available).

In [ ]:
from frothiq.features.pipeline import load_gold

gold = load_gold(ROOT / 'data' / 'processed' / 'gold.parquet')
feature_cols = list_feature_cols(gold, target_cols=TARGET_COLS)
current_row = gold.dropna(subset=feature_cols).iloc[-1]

print(f'Current pH: {current_row["ore_pulp_ph"]:.3f}')
print(f'Current density: {current_row["ore_pulp_density"]:.3f}')
print(f'Current amina flow: {current_row["amina_flow"]:.1f}')

# Cast to float64: heterogeneous Series → DataFrame leaves dtype=object,
# which LightGBM rejects. Explicit cast is the standard fix.
X_current = current_row[feature_cols].to_frame().T.astype("float64")
current_pred = float(model.predict(X_current)[0])
print(f'\nCurrent predicted {target}: {current_pred:.3f}')
print(f'Actual measured (lab): {current_row[target]:.3f}')

## 3. Naive what-if: change pH by ±0.5 and see Δ in predicted % Iron

In [ ]:
ph_deltas = np.linspace(-0.5, 0.5, 11)
results_naive = []
for d in ph_deltas:
    new_ph = current_row['ore_pulp_ph'] + d
    res = simulate_whatif_naive(
        model, current_row, feature_cols,
        overrides={'ore_pulp_ph': new_ph},
    )
    results_naive.append({
        'delta_ph': float(d),
        'new_ph': float(new_ph),
        'baseline_pred': float(res.baseline_pred[0]),
        'cf_pred': float(res.counterfactual_pred[0]),
        'delta_pred': float(res.delta[0]),
    })
df_naive = pd.DataFrame(results_naive)
df_naive.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_naive['delta_ph'], df_naive['delta_pred'], 'o-', color='#1f77b4')
ax.axhline(0, color='gray', ls=':')
ax.axvline(0, color='gray', ls=':')
ax.set_xlabel('Δ pH (override)')
ax.set_ylabel(f'Δ predicted {target}')
ax.set_title(f'Naive what-if: sensitivity of {target} to pH change')
fig.tight_layout()

## 4. Exact what-if: same query, recomputing all derived features

For the exact path we need the recent window of **raw** sensor data, not the gold features. We rebuild from the original CSV.

In [ ]:
data = load_flotation()
raw = data.df

# Find the timestamp of our 'current_row' in the raw data and pull a 540-row window ending there.
current_ts = current_row['timestamp']
# Largest rolling window in our pipeline is 540 rows.
WINDOW = 540
raw_idx = raw[raw['timestamp'] <= current_ts].index[-1]
raw_window = raw.iloc[max(0, raw_idx - WINDOW + 1) : raw_idx + 1].reset_index(drop=True)
print(f'Recent window: {len(raw_window)} rows ending at {raw_window.timestamp.iloc[-1]}')

In [ ]:
results_exact = []
for d in ph_deltas:
    new_ph = current_row['ore_pulp_ph'] + d
    res = simulate_whatif_exact(
        model, raw_window,
        sensor_cols=data.sensor_cols, feature_cols=feature_cols,
        overrides={'ore_pulp_ph': new_ph},
    )
    results_exact.append({
        'delta_ph': float(d),
        'cf_pred_exact': float(res.counterfactual_pred[0]),
        'delta_pred_exact': float(res.delta[0]),
    })
df_exact = pd.DataFrame(results_exact)
df_compare = df_naive.merge(df_exact, on='delta_ph')
df_compare.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_compare['delta_ph'], df_compare['delta_pred'], 'o-', label='naive', color='#1f77b4')
ax.plot(df_compare['delta_ph'], df_compare['delta_pred_exact'], 's--', label='exact (recompute)', color='#ff7f0e')
ax.axhline(0, color='gray', ls=':')
ax.axvline(0, color='gray', ls=':')
ax.set_xlabel('Δ pH (override)')
ax.set_ylabel(f'Δ predicted {target}')
ax.set_title('What-if: naive vs exact')
ax.legend()
fig.tight_layout()

## 5. Production guidance

* **Naive** for real-time UI (operator drags a slider). Sub-millisecond per evaluation.
* **Exact** for committed decisions (operator clicks 'Apply'). ~100 ms per evaluation.
* If the difference between naive and exact is large, the operator's intuition is misleading — show both and the model's confidence interval if available.

## What's next

* Wire the simulator into the Streamlit dashboard (`src/frothiq/serving/dashboard.py`).
* Wire the FastAPI service (`src/frothiq/serving/api.py`) — `POST /whatif`.
* For Databricks deployment, see `docs/databricks_deploy.md`.